# Data Source 03 · 제조·품질

> **솔직한 한계:** 달력 시간이 있는 무료 제조 시계열은 드물다. 아래 후보는 이 저장소에서 검증하지 않았다(검증 C).

| 출처 | 비고 |
|---|---|
| UCI SECOM (id 179) | 센서 + 합불 라벨. 타임스탬프 제공 여부·형식은 직접 확인 |
| NASA C-MAPSS / PHM08 | 사이클 단위 열화 자료. 달력 시간 아님. 라이선스 확인 필요 |
| 합성 공정 시계열 | 관리도·변화점 방법 시연용. 실제 자료 아님 |

이 노트북의 기본 경로는 **합성 공정 자료 + 선택적 SECOM 시도**이다. SECOM 시도는 `ucimlrepo` 설치와 네트워크가 필요하다.

In [ ]:
import io, json, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt

def fetch_bytes(url, timeout=30):
    """무키 공개 URL 요청. 실패 시 예외 -> 호출부에서 합성 대체."""
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def synthetic_series(n, freq, seed, trend=0.05, season=12, noise=1.0, start="2015-01-01"):
    """NOT real data: 방법 시연용 합성 시계열."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    y = 100 + trend * t + 3 * np.sin(2 * np.pi * t / season) + rng.normal(0, noise, n)
    return pd.Series(y, index=pd.date_range(start, periods=n, freq=freq), name="value")

def quality_report(s):
    s = s.sort_index()
    print("관측 수:", len(s), "| 기간:", s.index.min(), "~", s.index.max())
    print("결측:", int(s.isna().sum()), "| 중복 시점:", int(s.index.duplicated().sum()))
    f = pd.infer_freq(s.index) if len(s) > 2 else None
    print("추정 주기:", f)

In [ ]:
def synthetic_process(n=400, seed=2026031, shift_at=250, shift=1.2):
    """NOT real data: 평균 이동이 있는 공정 측정값 (관리도 시연용)."""
    rng = np.random.default_rng(seed)
    x = rng.normal(10, 0.5, n)
    x[shift_at:] += shift
    return pd.Series(x, index=pd.RangeIndex(n), name="measure"), shift_at

In [ ]:
USE_SECOM = False   # True로 바꾸면 UCI SECOM을 시도 (pip install ucimlrepo 필요, 구조는 직접 확인)
DATA_KIND = None
if USE_SECOM:
    try:
        from ucimlrepo import fetch_ucirepo
        secom = fetch_ucirepo(id=179)
        X = secom.data.features
        print("SECOM 특성 shape:", X.shape, "| 타임스탬프 열 여부는 secom.data.original 확인")
        DATA_KIND = "REAL: UCI SECOM (구조 확인 필요)"
    except Exception as e:
        print("SECOM 수집 실패 -> 합성 대체:", type(e).__name__, e)
if DATA_KIND is None:
    y, brk = synthetic_process()
    DATA_KIND = "SYNTHETIC (NOT real data)"
    mu0, sd0 = y.iloc[:brk].mean(), y.iloc[:brk].std()
    fig, ax = plt.subplots()
    ax.plot(y.index.to_numpy(), y.to_numpy()); ax.set_title(DATA_KIND)
    ax.axhline(mu0 + 3 * sd0, ls="--"); ax.axhline(mu0 - 3 * sd0, ls="--"); plt.show()
print("DATA_KIND:", DATA_KIND)

## 점검 체크리스트
- 관측 단위가 **시간(달력)**인지, 로트·사이클 번호인지: 사이클 자료에 달력 기반 ARIMA/계절성을 적용하지 않는다
- 라벨(합불)의 시점 정렬과 누수 여부 (Ch.8)
- 센서 결측·드리프트와 교정 시점
- 라이선스 확인 전 원자료 재배포 금지